In [2]:
import probablepeople as pp 
import os
import sys
from pyspark.sql.window import Window
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

In [3]:

# 1. Point directly to your installed JDK 17 path
os.environ["JAVA_HOME"] = r"C:\Program Files\Eclipse Adoptium\jdk-17.0.20.8-hotspot"
os.environ["PATH"] = os.path.join(os.environ["JAVA_HOME"], "bin") + os.pathsep + os.environ["PATH"]

# Ensure PySpark uses the active Python interpreter
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

# 2. Now import and start Spark


spark = SparkSession.builder \
    .appName("pyspark_ylep") \
    .master("local[*]") \
    .config("spark.driver.memory", "10g") \
    .config("spark.driver.maxResultSize", "4g") \
    .config("spark.sql.shuffle.partitions", "24") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .config("spark.sql.autoBroadcastJoinThreshold", "64MB") \
    .getOrCreate()
print("Spark initialized successfully!")

c:\Users\User\AppData\Local\Programs\Python\Python312\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Spark initialized successfully!


USER_NAME

In [4]:
df_read_user_data = spark.read.json("D:\\A_Data_engineer\\Project_Yelp\\Yelp JSON\\yelp_academic_dataset_user.json")

In [5]:
df_user_remove_junk_symbols = df_read_user_data.filter(
    # Must contain at least one English letter OR CJK character
    (df_read_user_data["name"].rlike(r"[a-zA-Z]") | df_read_user_data["name"].rlike(r"[\u4E00-\u9FFF\u3040-\u309F\u30A0-\u30FF]"))
    &
    # AND must NOT contain numbers or junk symbols
    (~df_read_user_data["name"].rlike(r"[0-9@#$%^&*()_+=\[\]{};:\"\\|<>/?]"))
)
df_user_remove_junk_symbols.cache()

DataFrame[average_stars: double, compliment_cool: bigint, compliment_cute: bigint, compliment_funny: bigint, compliment_hot: bigint, compliment_list: bigint, compliment_more: bigint, compliment_note: bigint, compliment_photos: bigint, compliment_plain: bigint, compliment_profile: bigint, compliment_writer: bigint, cool: bigint, elite: string, fans: bigint, friends: string, funny: bigint, name: string, review_count: bigint, useful: bigint, user_id: string, yelping_since: string]

In [6]:
#In this step, I verify whether the name is valid using probablepeople
list_names=[]
for row in df_user_remove_junk_symbols.select("name").distinct().collect():
    parsed = pp.parse(row.name)
    list_names.append(parsed) 

In [7]:
#Identify which names are not permitted in the system
unauthorized_username={"CorporationNameAndCompany","CorporationName","CorporationNameOrganization","CorporationLegalType","CorporationCommitteeType","CorporationNameBranchType","CorporationNameBranchIdentifier"}

In [8]:
valid_username=[]
for row in list_names:
  username=""
  count=0
  for name,username_type in row:
      if username_type not in unauthorized_username:
         username=username+""+name
         count=count+1
     

  valid_username.append(username) 

In [9]:
df_valid_username=df_user_remove_junk_symbols.filter(df_user_remove_junk_symbols["name"].isin(valid_username)) # df_user_remove_junk_symbols
df_valid_username.cache()

DataFrame[average_stars: double, compliment_cool: bigint, compliment_cute: bigint, compliment_funny: bigint, compliment_hot: bigint, compliment_list: bigint, compliment_more: bigint, compliment_note: bigint, compliment_photos: bigint, compliment_plain: bigint, compliment_profile: bigint, compliment_writer: bigint, cool: bigint, elite: string, fans: bigint, friends: string, funny: bigint, name: string, review_count: bigint, useful: bigint, user_id: string, yelping_since: string]

Join_Date

In [10]:
df_valid_date = df_valid_username.filter(
    F.to_date(F.col("yelping_since")).between("2004-10-01", F.current_date())
).withColumn("Join_Date", F.to_date(F.col("yelping_since"))).drop(F.col("yelping_since"))

In [11]:
df_valid_review_number = df_valid_date.filter((F.col("useful").try_cast("int") >=0)
                     & (F.col("funny").try_cast("int") >=0) 
                     & (F.col("cool").try_cast("int") >=0) 
                     & (F.col("fans").try_cast("int") >=0)
                     & (F.col("compliment_hot").try_cast("int") >=0)
                     & (F.col("compliment_more").try_cast("int") >=0) 
                     & (F.col("compliment_profile").try_cast("int") >=0) 
                     & (F.col("compliment_cute").try_cast("int") >=0)
                     & (F.col("compliment_list").try_cast("int") >=0)
                     & (F.col("compliment_note").try_cast("int") >=0) 
                     & (F.col("compliment_plain").try_cast("int") >=0) 
                     & (F.col("compliment_cool").try_cast("int") >=0)
                     & (F.col("compliment_funny").try_cast("int") >=0)
                     & (F.col("compliment_writer").try_cast("int") >=0) 
                     & (F.col("compliment_photos").try_cast("int") >=0) 
                     & (F.col("average_stars").try_cast("float") >=0)
                     & (F.col("review_count").try_cast("int") >=0))

In [12]:
def try_cast(val, target_type, default=None):
    try:
        return target_type(val)
    except (ValueError, TypeError):
        return default
    
def clean_address(text):
    first="has no elite year"
    last="has no elite year"
    if text is None:
        return first,last,0
    else:
        list_date=[]
        text_split = text.split(",")
        for i in text_split:
            if try_cast(i, int):
                if int(i) >= 2004 :
                    list_date.append(int(i))
            else:
                text_split.remove(i)
        list_date.sort()
        if len(list_date)>0:
            first=str(list_date[0])
            last=str(list_date[-1])
            
    return first ,last , len(list_date)
# define the schema for 
schema = StructType([
    StructField("first_elite_year", StringType(), True),
    StructField("last_elite_year", StringType(), True),
     StructField("count_elite_years", StringType(), True)
])            
clean_address_udf = F.udf(clean_address, schema)

df_elite = (
    df_valid_review_number.withColumn("new_eilte", clean_address_udf(F.col("elite")))
      .withColumn("first_elite_year", F.col("new_eilte.first_elite_year"))
      .withColumn("last_elite_year", F.col("new_eilte.last_elite_year"))
      .withColumn("user_elite_count", F.col("new_eilte.count_elite_years"))
      .drop("new_eilte",F.col("elite"))
)  


c:\Users\User\AppData\Local\Programs\Python\Python312\Lib\site-packages\pyspark\sql\udf.py:116: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
c:\Users\User\AppData\Local\Programs\Python\Python312\Lib\site-packages\pyspark\sql\udf.py:120: RuntimeWarning: Arrow optimization failed to enable because PyArrow or Pandas is not installed. Falling back to a non-Arrow-optimized UDF.
  warnings.warn(


In [13]:
df_add_surrogate_key = df_elite.withColumn("User_pk_sk", F.monotonically_increasing_id() + 1)

In [14]:
df_user = df_add_surrogate_key.select(
    F.col("User_pk_sk"),
    F.col("User_id"),
    F.col("name").alias("User_name"),   
    F.col("first_elite_year"),
    F.col("last_elite_year"),
    F.col("user_elite_count"),
    F.col("Join_Date").alias("join_date"),
    F.col("review_count").alias("User_review_count"),
    F.col("useful").alias("review_useful"),
    F.col("funny").alias("review_funny"),
    F.col("cool").alias("review_cool"),
    F.col("fans").alias("review_fans"),
    F.col("average_stars").alias("review_average_stars"), 
    F.col("compliment_hot").alias("review_compliment_hot"),
    F.col("compliment_more").alias("review_compliment_more"),
    F.col("compliment_profile").alias("review_compliment_profile"),
    F.col("compliment_cute").alias("review_compliment_cute"),
    F.col("compliment_list").alias("review_compliment_list"),
    F.col("compliment_note").alias("review_compliment_note"),
    F.col("compliment_plain").alias("review_compliment_plain"),
    F.col("compliment_cool").alias("review_compliment_cool"),
    F.col("compliment_funny").alias("review_compliment_funny"),
    F.col("compliment_writer").alias("review_compliment_writer"),
    F.col("compliment_photos").alias("review_compliment_photos")
)

In [15]:
df_user.show()

+----------+--------------------+---------+-----------------+-----------------+----------------+----------+-----------------+-------------+------------+-----------+-----------+--------------------+---------------------+----------------------+-------------------------+----------------------+----------------------+----------------------+-----------------------+----------------------+-----------------------+------------------------+------------------------+
|User_pk_sk|             User_id|User_name| first_elite_year|  last_elite_year|user_elite_count| join_date|User_review_count|review_useful|review_funny|review_cool|review_fans|review_average_stars|review_compliment_hot|review_compliment_more|review_compliment_profile|review_compliment_cute|review_compliment_list|review_compliment_note|review_compliment_plain|review_compliment_cool|review_compliment_funny|review_compliment_writer|review_compliment_photos|
+----------+--------------------+---------+-----------------+-----------------+---

In [17]:


# Repartition the DataFrame
df_repartitioned = df_user.repartition(5)  # Adjust the number based on your cluster size
# Save the repartitioned DataFrame
df_repartitioned.write.mode("overwrite").parquet("D:/A_Data_engineer/Project_Yelp/store_dimesnion_user")

Py4JJavaError: An error occurred while calling o130988.parquet.
: java.lang.RuntimeException: java.io.FileNotFoundException: java.io.FileNotFoundException: HADOOP_HOME and hadoop.home.dir are unset. -see https://cwiki.apache.org/confluence/display/HADOOP2/WindowsProblems
	at org.apache.hadoop.util.Shell.getWinUtilsPath(Shell.java:790)
	at org.apache.hadoop.util.Shell.getSetPermissionCommand(Shell.java:299)
	at org.apache.hadoop.util.Shell.getSetPermissionCommand(Shell.java:315)
	at org.apache.hadoop.fs.RawLocalFileSystem.setPermission(RawLocalFileSystem.java:1179)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkOneDirWithMode(RawLocalFileSystem.java:861)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirsWithOptionalPermission(RawLocalFileSystem.java:901)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirs(RawLocalFileSystem.java:873)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirsWithOptionalPermission(RawLocalFileSystem.java:900)
	at org.apache.hadoop.fs.RawLocalFileSystem.mk